# Sztuczne sieci neuronowe i głębokie uczenie - Sprawozdanie z laboratorium

Imie i nazwisko: Przemysław Kostrzewa

## Temat:
Praca z korpusami językowymi

### Cel ćwiczenia:
Celem tego laboratorium jest dowiedzenie się, czym jest korpus językowy i dlaczego jest fundamentem NLP. Ponadto, celem jest też nauczenie się znajdowania i pobierania gotowych korpusów językowych: HuggingFace Hub. Następnie przeprowadzone zostaną podstawowe analizy statystyczne tekstu.

### Wykorzystane narzędzia:
Python 3.x, Google Colab / Jupyter, oraz biblioteki: datasets, transformers, tokenizers, matplotlib, numpy, pandas, re, nltk.

### Zadanie 1: Pierwsza eksploracja korpusu

**Przebieg ćwiczenia:**
W tej części zapoznano się z pobranym korpusem 200 artykułów polskiej Wikipedii. Wyświetlono tytuły, liczbę słów i pierwsze dwa zdania dla pierwszych 10 artykułów oraz znaleziono najkrótszy i najdłuższy z nich.

**Czym różni się korpus językowy od zbioru danych do klasyfikacji?**
Korpus językowy to duży zbiór tekstów zebranych w sposób zorganizowany i celowy, który działa jako materiał empiryczny do badań i baza treningowa dla modeli od podstaw. Z kolei zbiór danych do klasyfikacji to z reguły mniejszy, węższy dziedzinowo zbiór, który został dodatkowo opatrzony adnotacjami i etykietami. Podczas gdy korpus pozwala modelowi uczyć się ogólnych statystyk i struktury samego języka (np. przewidywania kolejnego słowa), zbiór klasyfikacyjny służy wyuczeniu modelu rozwiązywania konkretnego zadania decyzyjnego, takiego jak ocena sentymentu czy kategoryzacja spamu.

In [ ]:
import re

print("--- Tytuły pierwszych 10 artykułów ---")
for i in range(10):
    print(f"{i+1}. {titles[i]}")

print("\n--- Szczegóły pierwszych 10 artykułów ---")
for i in range(10):
    words = texts[i].split()
    # Prosty podział na zdania z użyciem wyrażeń regularnych
    sentences = re.split(r'(?<=[.!?])\s+', texts[i].strip())
    first_two_sentences = " ".join(sentences[:2])
    print(f"Tytuł: {titles[i]}")
    print(f"Liczba słów: {len(words)}")
    print(f"Pierwsze 2 zdania: {first_two_sentences}\n")

# Znajdowanie najkrótszego i najdłuższego artykułu
lengths = [len(t.split()) for t in texts]
min_idx = lengths.index(min(lengths))
max_idx = lengths.index(max(lengths))

print(f"Najkrótszy artykuł: '{titles[min_idx]}' ({min(lengths)} słów)")
print(f"Najdłuższy artykuł: '{titles[max_idx]}' ({max(lengths)} słów)")

### Zadanie 2: Statystyki leksykalne i prawo Zipfa

**Przebieg ćwiczenia:**
Połączono wszystkie teksty z korpusu, dokonano prostej tokenizacji, a następnie policzono tokeny i unikalne słowa (types) wyznaczając TTR (Type-Token Ratio). Znaleziono najczęstsze i najrzadsze słowa oraz narysowano wykres prawa Zipfa.

**Co prawo Zipfa oznacza praktycznie dla budżetu tokenów w LLM?**
Prawo Zipfa pokazuje, że częstotliwość słowa w korpusie jest odwrotnie proporcjonalna do jego rangi, co oznacza, że rozkład słów jest wysoce nierówny. Zaledwie garstka słów (np. "w", "i", "na") pojawia się ogromną liczbę razy. Dla budżetu tokenów w modelach LLM oznacza to, że lwią część okna kontekstowego "pożerają" słowa powszechne o niskiej wartości semantycznej. Zrozumienie tego zjawiska pozwala zoptymalizować budowę słowników w modelach językowych (np. za pomocą podziału na podsłowa), aby model efektywnie kodował rzadkie, ale ważne słowa, nie zapełniając okna wyłącznie najpopularniejszymi formami.

In [ ]:
import re
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np

# Połączenie tekstów, usunięcie interpunkcji i lowercasing
all_text = " ".join(texts)
cleaned_text = re.sub(r'[^\w\s]', '', all_text).lower()
tokens = cleaned_text.split()

total_tokens = len(tokens)
types = set(tokens)
unique_tokens = len(types)
# Type-Token Ratio (TTR) to stosunek liczby unikalnych słów do łącznej liczby tokenów
ttr = unique_tokens / total_tokens if total_tokens > 0 else 0

print(f"Łączna liczba tokenów: {total_tokens}")
print(f"Liczba unikalnych tokenów (types): {unique_tokens}")
print(f"TTR (Type-Token Ratio): {ttr:.4f}")

# Licznik częstotliwości
freq = Counter(tokens)
most_common = freq.most_common(20)
least_common = freq.most_common()[:-21:-1] # Pobieramy 20 od końca

print("\n20 najczęstszych słów:")
for w, c in most_common:
    print(f"{w}: {c}")

print("\n20 najrzadziej występujących słów:")
for w, c in least_common:
    print(f"{w}: {c}")

# Wykres prawa Zipfa
sorted_counts = [count for word, count in freq.most_common()]
ranks = np.arange(1, len(sorted_counts) + 1)

plt.figure(figsize=(10, 5))
plt.loglog(ranks, sorted_counts, marker='.', linestyle='none', color='tomato')
plt.title('Prawo Zipfa - Wykres log-log')
plt.xlabel('Log (ranga)')
plt.ylabel('Log (częstotliwość)')
plt.grid(True)
plt.show()

### Zadanie 3: Rozkład długości artykułów

**Przebieg ćwiczenia:**
Obliczono długość każdego artykułu w słowach, wyznaczono min, max, średnią, medianę, odchylenie standardowe oraz 95. percentyl (P95). Narysowano histogram rozkładu długości.

**Gdybyśmy chcieli przetworzyć artykuły tokenizatorem BERT (max_length=512 tokenów), ile artykułów wymagałoby obcięcia? Czy to problem?**
Przyjęcie średnio 1.3 tokenu na polskie słowo oznacza, że limit okna wynoszący `max_length = 512` tokenów pozwala wczytać około 393 słowa. Według statystyk w naszym korpusie znaczna część artykułów przekracza ten próg i musiałaby zostać obcięta. Takie zjawisko to spory problem – prowadzi do utraty informacji znajdujących się w dalszej części artykułu, co dla modeli oznacza brak pełnego kontekstu analizowanego tekstu (szczególnie np. podsumowań czy ważnych konkluzji).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

lengths = [len(t.split()) for t in texts]

min_len = np.min(lengths)
max_len = np.max(lengths)
mean_len = np.mean(lengths)
median_len = np.median(lengths)
std_len = np.std(lengths)
p95_len = np.percentile(lengths, 95)

print(f"Min: {min_len}")
print(f"Max: {max_len}")
print(f"Średnia: {mean_len:.1f}")
print(f"Mediana: {median_len:.1f}")
print(f"Odchylenie standardowe: {std_len:.1f}")
print(f"P95: {p95_len:.1f}")

plt.figure(figsize=(12, 6))
plt.hist(lengths, bins=30, color='steelblue', edgecolor='black')
plt.axvline(median_len, color='red', linestyle='dashed', linewidth=2, label=f'Mediana: {median_len:.1f}')
plt.axvline(p95_len, color='orange', linestyle='dashed', linewidth=2, label=f'P95: {p95_len:.1f}')
plt.title("Rozkład długości tekstów")
plt.xlabel("Liczba słów")
plt.ylabel("Liczba artykułów")
plt.legend()
plt.show()

# Oszacowanie ucięć dla HerBERT (max 512 tokenów)
max_words_limit = 512 / 1.3
truncated_count = sum(1 for l in lengths if l > max_words_limit)
print(f"\nLimit około {max_words_limit:.0f} słów dla 512 tokenów.")
print(f"Liczba artykułów wymagających obcięcia: {truncated_count} ({truncated_count/len(lengths)*100:.1f}%)")

### Zadanie 4: Tokenizacja polska a angielska

**Przebieg ćwiczenia:**
Sprawdzono, jak tokenizatory radzą sobie z językiem polskim. Wybrano 10 zdań i przetworzono je dwoma tokenizatorami: `allegro/herbert-base-cased` oraz `bert-base-multilingual-cased`. Wyniki zestawiono w tabeli i zaprezentowano na wykresie słupkowym.

**Który model jest bardziej 'efektywny' dla polskiego i dlaczego? Co oznacza większy/mniejszy słownik podsłów dla jakości modelu?**
W przypadku języka bogatego morfologicznie, takiego jak język polski, zdecydowanie efektywniejszy jest model dedykowany – w tym przypadku **HerBERT**. Został on wytrenowany na polskim korpusie tekstowym, dzięki czemu zawiera bardzo dużo typowych polskich słów, form i przyrostków w swoim słowniku podsłów. Model **mBERT** jest wielojęzyczny, dlatego z braku miejsca w słowniku często dzieli polskie słowa na znacznie mniejsze, nienaturalne fragmenty (tzw. over-tokenization). Duży rozmiar trafnego słownika podsłów wpływa krytycznie na jakość i wydajność działania: pozwala na głębszą, łatwiejszą w nauce reprezentację (jedno sensowne podsłowo zamiast wielu znakowych) i pozwala na przetwarzanie dłuższego tekstu w ograniczonym oknie modelowym (zmniejsza ryzyko przekroczenia limitu max_length).

In [ ]:
import pandas as pd
import re
from transformers import AutoTokenizer
import matplotlib.pyplot as plt

herbert_tokenizer = AutoTokenizer.from_pretrained("allegro/herbert-base-cased")
mbert_tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")

# Pobieramy 10 pierwszych zdań z pierwszego artykułu w korpusie
sentences = re.split(r'(?<=[.!?])\s+', texts[0].strip())[:10]

data = []
for i, sentence in enumerate(sentences):
    herbert_tokens = herbert_tokenizer.tokenize(sentence)
    mbert_tokens = mbert_tokenizer.tokenize(sentence)

    data.append({
        "zdanie_etykieta": f"Zdanie {i+1}",
        "zdanie": sentence,
        "herbert_n": len(herbert_tokens),
        "mbert_n": len(mbert_tokens),
        "roznica": len(mbert_tokens) - len(herbert_tokens)
    })

    print(f"Zdanie {i+1}: {sentence}")
    print(f"HerBERT: {herbert_tokens} (Suma: {len(herbert_tokens)})")
    print(f"mBERT: {mbert_tokens} (Suma: {len(mbert_tokens)})\n")

df = pd.DataFrame(data)

# Wizualizacja na wykresie
df.plot(x='zdanie_etykieta', y=['herbert_n', 'mbert_n'], kind='bar', figsize=(12, 6), color=['#2ca02c', '#1f77b4'])
plt.title('Liczba tokenów dla zdań: HerBERT vs mBERT')
plt.ylabel('Liczba tokenów')
plt.xlabel('Kolejne zdania')
plt.xticks(rotation=0)
plt.legend(['HerBERT', 'mBERT'])
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

# Opcjonalne wyświetlenie samej tabeli różnic
display(df[['zdanie_etykieta', 'herbert_n', 'mbert_n', 'roznica']])

### Zadanie 5: N-gramy i kolokacje

**Przebieg ćwiczenia:**
Przeprowadzono analizę bigramów i trigramów w korpusie, wyszukując najczęstsze związki wyrazowe. W części C odfiltrowano zbiór usuwając tzw. stop words (słowa z języka polskiego).

**Jak stopwords wpływają na sensowność kolokacji? Kiedy je usuwać, a kiedy zostawić?**
W surowym zbiorze najwyższe pozycje zajmują zbitki z przyimkami i spójnikami (np. "w roku", "na świecie"). Dopiero odfiltrowanie *stop words* ujawnia nam użyteczne i sensowne związki wyrazowe niosące czystą wartość znaczeniową i tematyczną (np. "stanów zjednoczonych", "druga wojna").
Usuwanie takich "pustych" słów ma kluczowe znaczenie w klasycznych metodach przetwarzania tekstu (np. ekstrakcja słów kluczowych czy analiza n-gramów), ponieważ pomaga "odszumić" tekst i skupić model na meritum. Nie należy jednak tego robić podczas przygotowywania wejścia dla nowoczesnych LLM (Large Language Models) lub algorytmów analizujących strukturę składniową (Dependency Parsing). W takich przypadkach usunięcie spójników całkowicie niszczy gramatyczny porządek zdań i powiązania logiczne, które są dla takich modeli krytyczne z punktu widzenia prawidłowego pojęcia relacji.

In [ ]:
import re
from collections import Counter
import nltk
from nltk.util import ngrams

# Zabezpieczenie przed brakiem pakietu NLTK na niektórych systemach
nltk.download('punkt', quiet=True)

# Przygotowanie prostych tokenów z wszystkich tekstów (podobnie jak w zadaniu 2)
all_text = " ".join(texts)
cleaned_tokens = re.sub(r'[^\w\s]', '', all_text).lower().split()

# --- CZĘŚĆ A: Bigramy ---
bigrams = list(ngrams(cleaned_tokens, 2))
bigram_freq = Counter(bigrams)
print("--- Część A: Top 20 bigramów ---")
for b, c in bigram_freq.most_common(20):
    print(f"{b}: {c}")

# --- CZĘŚĆ B: Trigramy ---
trigrams = list(ngrams(cleaned_tokens, 3))
trigram_freq = Counter(trigrams)
print("\n--- Część B: Top 15 trigramów ---")
for t, c in trigram_freq.most_common(15):
    print(f"{t}: {c}")

# --- CZĘŚĆ C: Bigramy bez stop słów ---
try:
    from stop_words import get_stop_words
    stopwords = set(get_stop_words('pl'))
except ImportError:
    print("\nBrak biblioteki stop_words. Uruchom: !pip install stop_words")
    stopwords = set()

# Filtrowanie i ponowne liczenie
filtered_tokens = [w for w in cleaned_tokens if w not in stopwords]
filtered_bigrams = list(ngrams(filtered_tokens, 2))
filtered_bigram_freq = Counter(filtered_bigrams)

print("\n--- Część C: Top 20 bigramów bez stop słów ---")
for b, c in filtered_bigram_freq.most_common(20):
    print(f"{b}: {c}")

### Wnioski

Podczas laboratorium zapoznano się z podstawowymi koncepcjami pracy z korpusami językowymi w Pythonie. Główne wnioski płynące z wykonanych ćwiczeń to:
* Korpusy różnią się od zbiorów klasyfikacyjnych swoim rozmiarem oraz przeznaczeniem – służą do fundamentalnego uczenia modeli na potężnych zasobach tekstu.
* Język naturalny charakteryzuje się bardzo nierównomiernym rozkładem słów (Prawo Zipfa), co należy brać pod uwagę przy projektowaniu słowników dla modeli LLM.
* Długość artykułów i zasób słów odgrywają istotną rolę w dobieraniu tokenizatorów i określeniu ograniczeń okna kontekstowego (max_length). Ucięcie zbyt długich tekstów prowadzi do utraty cennych informacji.
* W kontekście języków o skomplikowanej morfologii (jak polski) kluczowe jest używanie odpowiednio dopasowanych narzędzi, takich jak dedykowany model HerBERT, który skuteczniej sobie z nimi radzi niż ogólne narzędzia wielojęzyczne.
* Filtrowanie mało znaczących słów (stop words) bywa przydatne podczas prostych analiz n-gramów i kolokacji, ale dla zaawansowanych modeli opartych na sieciach neuronowych (LLM) zachowanie spójności gramatycznej zdań w pełnej formie jest niezbędne.